# 01. 데이터 준비

OSV-5M에서 20개국 이미지를 국가당 1,000장(train 680 / val 120 / test 200) 내려받는다.

- 분할 CSV(`data/splits/{train,val,test}.csv`)와 이미지 위치 색인(`data/splits/image_index.csv`)은 저장소에 포함되어 있다. 이 노트북은 기본적으로 그 파일을 읽어 이미지만 내려받는다.
- `REBUILD = True`로 바꾸면 메타데이터(train.csv 약 2.9GB)를 받아 분할과 색인을 처음부터 다시 만든다. seed가 고정되어 있어 같은 결과가 나온다.
- 이미지는 짧은 변을 224px로 줄여(비율 유지) Drive의 `aix-osv5m/images.zip`에 저장한다. 런타임: CPU로 충분하다.

In [ ]:
# 공통 셋업: 저장소 clone, 패키지 설치, Drive 연결
import os, sys
REPO_URL = "https://github.com/Cranezero/aix-assignment-pr.git"
REPO_DIR = "/content/aix-assignment-pr"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd
from src import config, osv5m

pd.set_option("display.max_rows", 50)
config.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(len(config.CODES), "개국:", config.CODES)

## 1. 국가 선정과 분할

국가 선정 기준
- 공식 test split에서 국가당 200장을 확보할 수 있을 것. 한 sequence(같은 사용자가 연속 촬영한 묶음)에서는 최대 5장만 뽑는다.
- 대륙을 고루 포함하고, 혼동이 예상되는 쌍(한국-일본, 스페인-포르투갈, 미국-캐나다, 노르웨이-스웨덴, 호주-남아공)을 포함한다.

분할 방법
- test: 공식 test split에서 추출. 공식 test는 train 이미지와 1km 이상 떨어져 있고 sequence를 공유하지 않는다.
- train/val: 공식 train split에서 추출해 85:15로 나눈다. 같은 sequence가 train과 val에 함께 들어가지 않게 sequence 단위로 나눈다.

In [ ]:
REBUILD = False

if REBUILD:
    meta_dir = "/content/meta"
    train_meta = osv5m.load_metadata(osv5m.download_metadata("train", meta_dir))
    test_meta = osv5m.load_metadata(osv5m.download_metadata("test", meta_dir))

    # 선정 국가의 공식 split 전체 장수
    stats = pd.DataFrame({
        "official_train": train_meta["country"].value_counts(),
        "official_test": test_meta["country"].value_counts(),
    }).loc[config.CODES]
    display(stats)

    splits = osv5m.make_splits(train_meta, test_meta, config.CODES, config.N_TRAIN, config.N_VAL,
                               config.N_TEST, seed=config.SEED, cap=config.SEQ_CAP)
    osv5m.check_splits(splits)
    osv5m.save_splits(splits, config.SPLITS_DIR)

    train_ids = set(splits["train"]["id"]) | set(splits["val"]["id"])
    index = pd.concat([osv5m.build_index("test", set(splits["test"]["id"])),
                       osv5m.build_index("train", train_ids)], ignore_index=True)
    index.sort_values(["split", "zip", "offset"]).to_csv(config.SPLITS_DIR / osv5m.INDEX_FILE, index=False)

In [ ]:
splits = {s: pd.read_csv(config.SPLITS_DIR / f"{s}.csv", dtype={"id": str, "sequence": str})
          for s in ["train", "val", "test"]}
osv5m.check_splits(splits)

table = pd.DataFrame({s: df["country"].value_counts() for s, df in splits.items()}).loc[config.CODES]
table.insert(0, "name", [config.COUNTRIES[c] for c in table.index])
display(table)
print({s: len(df) for s, df in splits.items()})

## 2. 이미지 내려받기

OSV-5M 이미지 zip은 압축 없이 저장되어 있다. 색인에 적힌 바이트 위치만 HTTP Range 요청으로 받으므로 train 전체(98개 zip, 약 246GB)를 받지 않아도 된다. 2만 장에 20\~40분 정도 걸린다.

In [ ]:
index = osv5m.load_index(config.SPLITS_DIR)
out_dir = config.LOCAL_IMAGE_DIR

for split in ["test", "train"]:
    part = index[index["split"] == split]
    failed = osv5m.download_images(split, part, out_dir)
    if failed:  # 일시적 네트워크 오류는 한 번 더 시도한다
        failed = osv5m.download_images(split, part[part["id"].isin(failed)], out_dir)
    print(split, "실패:", len(failed))
    assert not failed

n_files = len(list(out_dir.glob("*.jpg")))
print("저장된 이미지:", n_files)
assert n_files == len(index)

In [ ]:
# Drive에 zip 하나로 저장한다 (jpg라 압축하지 않음)
import zipfile
from tqdm.auto import tqdm
from src.data import IMAGES_ZIP

zip_path = config.DRIVE_DIR / IMAGES_ZIP
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as z:
    for p in tqdm(sorted(out_dir.glob("*.jpg"))):
        z.write(p, arcname=p.name)
print(zip_path, round(zip_path.stat().st_size / 1e6), "MB")

In [ ]:
# 확인: 이미지 크기 분포 (짧은 변 224)
from PIL import Image
sizes = pd.DataFrame([Image.open(p).size for p in sorted(out_dir.glob("*.jpg"))[:2000]], columns=["w", "h"])
print(sizes.describe().round(1))